<a href="https://colab.research.google.com/github/ayman-06-stack/BIGDATALABS/blob/main/Lab_Spark_Streaming/Streaming_spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [4]:
import os

In [5]:
if not os.path.exists('/content/mon_flux'):
    os.makedirs('/content/mon_flux')

In [9]:
with open('/content/mon_flux/test.txt', 'w') as f:
    f.write("spark streaming dstream spark")

In [25]:
import socket, time, threading
from pyspark import SparkContext
from pyspark.streaming import StreamingContext

# --- PARTIE 1 : Le Serveur (Simulateur de flux) ---
def start_socket_server():
    host = "localhost"
    port = 9999
    s = socket.socket()
    s.bind((host, port))
    s.listen(1)
    conn, addr = s.accept()
    messages = ["spark streaming dstream", "spark spark streaming", "big data spark"]
    try:
        while True:
            for msg in messages:
                conn.send((msg + "\n").encode())
                time.sleep(2)
    except:
        conn.close()

# Lancer le serveur en arrière-plan
threading.Thread(target=start_socket_server, daemon=True).start()

# --- PARTIE 2 : Le Client Spark Streaming ---
sc = SparkContext.getOrCreate()
ssc = StreamingContext(sc, 5)

# Lecture via le port 9999
lines = ssc.socketTextStream("localhost", 9999)

# Traitement WordCount
words = lines.flatMap(lambda line: line.split(" "))
pairs = words.map(lambda w: (w, 1))
counts = pairs.reduceByKey(lambda a, b: a + b)

counts.pprint()

# Lancement
ssc.start()
ssc.awaitTerminationOrTimeout(30)
ssc.stop(stopSparkContext=False)

Exception in thread Thread-21 (start_socket_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-439579536.py", line 10, in start_socket_server
OSError: [Errno 98] Address already in use


-------------------------------------------
Time: 2026-01-16 20:17:15
-------------------------------------------
('streaming', 2)
('dstream', 1)
('spark', 3)

-------------------------------------------
Time: 2026-01-16 20:17:20
-------------------------------------------
('big', 1)
('streaming', 2)
('dstream', 1)
('data', 1)
('spark', 4)

-------------------------------------------
Time: 2026-01-16 20:17:25
-------------------------------------------
('big', 1)
('streaming', 1)
('dstream', 1)
('data', 1)
('spark', 2)

-------------------------------------------
Time: 2026-01-16 20:17:30
-------------------------------------------
('streaming', 2)
('big', 1)
('dstream', 1)
('spark', 4)
('data', 1)

-------------------------------------------
Time: 2026-01-16 20:17:35
-------------------------------------------
('streaming', 1)
('big', 1)
('spark', 3)
('data', 1)

-------------------------------------------
Time: 2026-01-16 20:17:40
-------------------------------------------
('streami